# Experimento: recorte completo versus segmentos, com fundo preservado

Hipótese: o treino somente normal pode separar anomalias sem rótulos de defeito no ajuste ou na calibração. O resultado deve ser medido por dataset/domínio; não pressupõe generalização universal.

Execute a instalação, disponibilize o pacote atualizado e configure o projeto como no notebook principal. O ZIP de código precisa incluir `crop_experiment.py`.


In [2]:
import importlib.metadata
import os
import subprocess
import sys

REQUIRED_PACKAGES = {
    "anomalib": "2.6.0",
    "ultralytics-opencv-headless": "8.4.123",
}

def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

needs_installation = any(
    installed_version(package) != version
    for package, version in REQUIRED_PACKAGES.items()
)

if needs_installation:
    print("Instalando o ambiente fixo do PatchCore...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade-strategy",
        "only-if-needed",
        "anomalib==2.6.0",
        "ultralytics-opencv-headless==8.4.123",
    ])
    print("Instalacao concluida. O kernel sera reiniciado uma unica vez.")
    os.kill(os.getpid(), 9)

print("Ambiente configurado. Continue para a proxima celula.")


Ambiente configurado. Continue para a proxima celula.


## 2. Disponibilizar o código ao kernel

No Colab, monte o Drive uma vez. A célula procura a pasta `insulator_anomaly` ou o ZIP de código em Meu Drive e em `Meu Drive/Colab_Notebooks`. Para outro local, preencha `CODE_PATH` (pasta que contém o pacote) ou `CODE_ZIP_PATH`.


In [3]:
from pathlib import Path
import sys
import importlib
import tempfile
import zipfile

CODE_PATH = ""
CODE_ZIP_PATH = ""
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")

if CODE_PATH and CODE_ZIP_PATH:
    raise ValueError("Preencha apenas CODE_PATH ou CODE_ZIP_PATH.")
code_roots = [Path(CODE_PATH)] if CODE_PATH else [Path.cwd()]
if IN_COLAB and not CODE_PATH:
    code_roots += [Path("/content/drive/MyDrive"), Path("/content/drive/MyDrive/Colab_Notebooks")]
code_root = None if CODE_ZIP_PATH else next(
    (root for root in code_roots if (root / "insulator_anomaly" / "__init__.py").is_file() and (root / "insulator_anomaly" / "crop_experiment.py").is_file() and (root / "insulator_anomaly" / "map_postprocessing.py").is_file()), None)
if code_root is None:
    code_archive = Path(CODE_ZIP_PATH) if CODE_ZIP_PATH else next(
        (root / "insulator_anomaly.zip" for root in code_roots if (root / "insulator_anomaly.zip").is_file()), None)
    if code_archive is None or not code_archive.is_file():
        raise FileNotFoundError("Disponibilize insulator_anomaly/ ou insulator_anomaly.zip ao kernel.")
    code_root = Path(tempfile.mkdtemp(prefix="insulator_code_")).resolve()
    with zipfile.ZipFile(code_archive) as bundle:
        if not {"insulator_anomaly/crop_experiment.py", "insulator_anomaly/map_postprocessing.py"}.issubset(bundle.namelist()):
            raise FileNotFoundError("ZIP antigo: envie o insulator_anomaly.zip atualizado, contendo crop_experiment.py.")
        for member in bundle.infolist():
            if not (code_root / member.filename).resolve().is_relative_to(code_root):
                raise ValueError(f"Caminho inválido no ZIP de código: {member.filename}")
        bundle.extractall(code_root)
    if not (code_root / "insulator_anomaly" / "__init__.py").is_file():
        raise FileNotFoundError("O ZIP de código não contém o pacote insulator_anomaly.")
for module_name in list(sys.modules):
    if module_name == "insulator_anomaly" or module_name.startswith("insulator_anomaly."):
        del sys.modules[module_name]
importlib.invalidate_caches()
sys.path.insert(0, str(code_root))
print("Código:", code_root)

import insulator_anomaly.crop_experiment as crop_module
print("Experimento carregado:", crop_module.__file__)


Código: /content/drive/MyDrive/Colab_Notebooks


### Conferir o código contra o ZIP do Drive
Ajuste DRIVE_ZIP. Opcionalmente informe o SHA256 gerado localmente para confirmar a versão enviada. Não atualiza módulos em execução. Se usar ZIP em /content, ajuste o caminho.

In [ ]:
"""Cole como célula após disponibilizar o código e antes de configurar o projeto."""

from pathlib import Path
import hashlib
import importlib.util
import sys
import zipfile

DRIVE_ZIP = Path("/content/drive/MyDrive/Colab_Notebooks/insulator_anomaly.zip")
# Opcional: hash impresso pelo empacotador local, para confirmar a versão enviada.
EXPECTED_ZIP_SHA256 = ""

if not DRIVE_ZIP.is_file():
    raise FileNotFoundError(f"Monte o Drive e confira o caminho: {DRIVE_ZIP}")

zip_hash = hashlib.sha256(DRIVE_ZIP.read_bytes()).hexdigest()
print("ZIP do Drive:", DRIVE_ZIP)
print("SHA256:", zip_hash)
if EXPECTED_ZIP_SHA256 and zip_hash != EXPECTED_ZIP_SHA256.strip().lower():
    raise RuntimeError("O ZIP do Drive não corresponde à versão local esperada. Atualize o arquivo.")

spec = importlib.util.find_spec("insulator_anomaly")
if spec is None or spec.origin is None:
    raise ModuleNotFoundError("Execute primeiro a célula que disponibiliza o código.")
package_dir = Path(spec.origin).parent
print("Pacote encontrado pelo kernel:", package_dir)

with zipfile.ZipFile(DRIVE_ZIP) as bundle:
    expected = {
        name.removeprefix("insulator_anomaly/"): bundle.read(name)
        for name in bundle.namelist()
        if name.startswith("insulator_anomaly/") and name.endswith(".py")
    }
    if not expected:
        raise ValueError("O ZIP não contém código insulator_anomaly/*.py.")
    missing = [name for name in expected if not (package_dir / name).is_file()]
    changed = [name for name, data in expected.items()
               if name not in missing and (package_dir / name).read_bytes() != data]
    extra = sorted({p.relative_to(package_dir).as_posix() for p in package_dir.rglob("*.py")} - set(expected))

if missing or changed or extra:
    print("Ausentes:", missing)
    print("Diferentes:", changed)
    print("Extras:", extra)
    raise RuntimeError(
        "Pacote diferente do ZIP do Drive. Reinicie a sessão, force CODE_ZIP_PATH "
        "para esse ZIP e execute novamente o carregador."
    )

print("OK: os arquivos do pacote correspondem ao ZIP disponível no Drive.")
if any(name.startswith("insulator_anomaly.") for name in sys.modules):
    print("Há módulos já importados. Se atualizou arquivos nesta sessão, reinicie: "
          "a comparação confere o disco, não o código antigo que pode estar na memória.")


## 3. Configurar o projeto

No Colab, o Python executa em outra máquina e não acessa `C:/...`. Deixe os campos vazios para procurar os dados junto ao notebook ou no Drive; preencha um deles para outra localização.

`ProjectConfig` centraliza parâmetros e caminhos. Construir a configuração não exige as imagens brutas, permitindo avaliar apenas os dados processados. `background_modes` pode selecionar os modos que deseja executar.


In [4]:
from insulator_anomaly.config import ProjectConfig
from insulator_anomaly.paths import resolve_project

PROJECT_PATH = ""  # Ex.: "/content/drive/MyDrive/Colab_Notebooks/anomalib_glass"
ZIP_PATH = ""      # Ex.: "/content/drive/MyDrive/anomalib_glass.zip"
project_root, persistent_root = resolve_project(PROJECT_PATH, ZIP_PATH, colab=IN_COLAB)
config = ProjectConfig(project_root, persistent_root)
print("Dados:", config.project_root)
print("Saídas persistentes:", config.persistent_results_root)
print("Modos:", config.background_modes)


Dados: /content/drive/MyDrive/Colab_Notebooks/anomalib_glass
Saídas persistentes: /content/drive/MyDrive/Colab_Notebooks/anomalib_glass/patchcore_insulators/results
Modos: ('original', 'blur', 'gray')


## 1. Definir o experimento

Use um destino novo por configuração. `whole` preserva o recorte inteiro; `segments` divide o eixo maior com sobreposição antes de redimensionar. Ambos preservam o fundo, sem máscara pixel a pixel, rotação ou YOLO implícito.

Por padrão, `AUTO_LOCALIZE=True` usa a união das caixas YOLO, sem usar as máscaras. Falta de detecção preserva a imagem inteira e é registrada em `localization.csv`. Inspecione também detecções parciais: a margem não garante recuperar objetos não detectados. Um JSON de caixas fornecido tem precedência. Para outro dataset, forneça seu localizador/caixas ou desative `AUTO_LOCALIZE`. Sem caixas e sem localização automática, a entrada é a imagem inteira: isso é uma referência sem segmentador, não uma localização automática da cadeia. Para estudar recortes com margem, forneça caixas de localização manuais ou automáticas em coordenadas da imagem original. Use o mesmo critério no treino e teste, sem escolher caixas a partir do defeito. Se houver várias cadeias, uma caixa deve englobar a região a preservar; este experimento não separa instâncias.

`groups.json` associa imagens da mesma cadeia/voo/sequência ao mesmo identificador. Um grupo já presente no teste não pode aparecer no treino. Sem esse arquivo, o holdout é por arquivo e não exclui vazamento visual.

Exemplo `boxes.json`: `{"train/good/foto.jpg": [100, 200, 1500, 600]}`. Exemplo `groups.json`: `{"train/good/foto.jpg": "voo_01_cadeia_03"}`. Os caminhos são relativos à pasta que contém `train/` e `test/`, com `/`.


In [8]:
from pathlib import Path
import json
import pandas as pd
from insulator_anomaly.crop_experiment import CropSettings, prepare_crop_experiment, run_crop_experiment

SOURCE_ROOT = config.dataset_root  # Pode apontar para qualquer dataset train/good, test/good, test/anomaly.
EXPERIMENT_ROOT = config.project_root / "experiments" / "crop_vs_segments_v1"
AUTO_LOCALIZE = True  # Usa caixas YOLO do projeto; False para imagens inteiras.
BOXES_JSON = ""   # Caminho para caixas; vazio preserva o quadro inteiro.
GROUPS_JSON = ""  # Preencher para imagens relacionadas.

settings = CropSettings(
    output_size=256,
    margin=0.20,             # Margem em cada lado, proporcional à dimensão da caixa.
    segment_aspect=1.5,      # Comprimento da janela / dimensão curta do recorte.
    overlap=0.25,
    validation_fraction=0.20,
    normal_fpr=0.05,         # Meta descritiva na validação normal; não é garantia no teste.
    seed=42,
)
boxes = json.loads(Path(BOXES_JSON).read_text(encoding="utf-8")) if BOXES_JSON else None
groups = json.loads(Path(GROUPS_JSON).read_text(encoding="utf-8")) if GROUPS_JSON else None
print("Origem:", SOURCE_ROOT)
print("Destino:", EXPERIMENT_ROOT)
print("Caixas:", len(boxes or {}), "Grupos informados:", len(groups or {}))

localization = None
if boxes is None and AUTO_LOCALIZE:
    from insulator_anomaly.crop_experiment import locate_crop_boxes
    boxes, localization = locate_crop_boxes(
        SOURCE_ROOT, config.yolo_model_path,
        confidence=config.yolo_confidence, class_ids=config.yolo_class_ids,
    )
    display(localization)


ModuleNotFoundError: No module named 'insulator_anomaly.crop_experiment'

## 2. Gerar e inspecionar

Execute uma vez. O gerador recusa sobrescrever destinos existentes. Para reabrir o mesmo experimento, carregue o manifesto pela célula seguinte. As origens históricas e checkpoints não são alterados. Confira principalmente os defeitos distantes antes de treinar.


In [ ]:
manifest = prepare_crop_experiment(
    SOURCE_ROOT, EXPERIMENT_ROOT, settings=settings, boxes=boxes, groups=groups,
    duplicate_policy="exclude_train",  # Preserva teste; registra exclusões sem alterar a origem.
)
display(manifest.groupby(["mode", "split", "y_true"]).agg(
    n_sources=("source_id", "nunique"), n_inputs=("image_path", "size"),
))

if localization is not None:
    localization.to_csv(EXPERIMENT_ROOT / "localization.csv", index=False)


In [ ]:
# Também pode ser executada ao reabrir um experimento já preparado.
import matplotlib.pyplot as plt
from PIL import Image
manifest = pd.read_csv(EXPERIMENT_ROOT / "manifest.csv")
source_id = manifest.loc[manifest.split == "test", "source_id"].iloc[0]
# Altere source_id para uma imagem distante ou defeituosa específica.
examples = manifest[manifest.source_id == source_id]
fig, axes = plt.subplots(1, len(examples), figsize=(4*len(examples), 4), squeeze=False)
for ax, row in zip(axes[0], examples.itertuples(), strict=True):
    with Image.open(row.image_path) as image:
        ax.imshow(image)
    ax.set_title(f"{row.mode}\n({row.x0}, {row.y0})–({row.x1}, {row.y1})")
    ax.axis("off")
plt.show()


## 3. Treinar e avaliar

Altere a chave abaixo para `True` depois de conferir os recortes. São dois modelos novos, ambos ajustados exclusivamente em `train/good`. A validação contém apenas normais reservadas por grupo antes dos recortes. Nenhuma anomalia participa do treino ou da escolha do limiar.

Usa scores sem normalização do pós-processador. O maior score dos segmentos define o score da origem. O limiar de cada modelo é o quantil superior dos scores de validação normal (decisão estrita `score > threshold`). Assim a calibração já considera o efeito de múltiplos segmentos. A taxa no teste pode diferir da meta, especialmente com poucos grupos de validação.

Logs do Anomalib são preservados. Checkpoints compatíveis podem ser reutilizados; mudanças nos arquivos preparados ou protocolo bloqueiam a reutilização. Execute em GPU. Os scores numéricos dos dois modelos não precisam ter a mesma escala.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    comparison, decisions = run_crop_experiment(EXPERIMENT_ROOT)
    display(comparison)
else:
    print("Confira os recortes e defina RUN_TRAINING=True para executar.")


## 4. Comparação final por imagem original

AP/AUROC avaliam ordenação; FP/FN/recall/precisão avaliam o limiar escolhido somente nas normais de validação. Uma imagem de teste defeituosa não torna todos os seus segmentos defeituosos: os rótulos só são usados após agregar por origem.

A mudança pareada abaixo identifica defeitos recuperados e normais que passaram a gerar alarme. Se só aumentar recall e também aumentar muito FP, não é ganho inequívoco. Para publicação, repita por grupos/sementes e reporte incerteza; generalização a domínios não vistos exige um experimento separado.


In [ ]:
comparison = pd.read_csv(EXPERIMENT_ROOT / "comparison.csv")
decisions = pd.read_csv(EXPERIMENT_ROOT / "test_decisions.csv")
display(comparison)
paired = decisions.pivot(index=["source_id", "y_true"], columns="mode", values="pred_label").reset_index()
paired["changed"] = paired["whole"] != paired["segments"]
display(paired[paired.changed])
print("Resultados:", EXPERIMENT_ROOT)


## B. Avaliação independente, usando apenas normais para calibrar

`run_crop_experiment` agora salva `map_manifest.csv` e os mapas de validação/teste. Se já treinou com uma versão anterior, execute novamente a etapa 3 de Crop_Comparison: reutiliza checkpoints compatíveis e refaz a inferência para exportar os mapas.

O manifesto requer mode, split, source_id, group_id, y_true, image_path e map_path. Pode fornecer defect_path com máscara binária real alinhada ao mapa; máscara YOLO do objeto não é anotação de defeito.

Divide os grupos normais de `val` em referência (fixar escalas) e calibração (decisão conformal). Também aceita splits explícitos reference/calibration/test. O usuário deve garantir que essas normais não participaram do treino. Não reserve referências a partir do teste histórico.

Calibração e alarme usam o máximo por grupo. Havendo várias fotos por grupo, o alarme do grupo é propagado às suas fotos; avalie principalmente unit=group. Sem grupos informados, cada imagem é um grupo e não se exclui vazamento visual. A interpretação conformal requer grupos normais independentes e permutáveis com os futuros grupos normais; não há garantia por pixel.

O score absolute é o top 1% do mapa, não o score reponderado do checkpoint. MEBin usa a intensidade original nos candidatos (zero fora deles); contraste usa o top 1% da resposta local. Essas agregações são hipóteses experimentais. Máscara visível pode existir sem alarme. Com poucos grupos normais, a resolução do p-valor pode impedir alarmes; o código avisa. AP/AUROC não escolhem o limiar. Não use teste para ajustar parâmetros e depois alegar avaliação independente.


In [ ]:
from datetime import datetime
from insulator_anomaly.map_postprocessing import MapSettings, evaluate_map_methods

RUN_MAP_EVALUATION = False  # True após exportar map_manifest.csv.
MAP_EXPERIMENT_ROOT = config.project_root / "experiments" / "crop_vs_segments_v1"
if RUN_MAP_EVALUATION:
    map_manifest = pd.read_csv(MAP_EXPERIMENT_ROOT / "map_manifest.csv")
    map_output = MAP_EXPERIMENT_ROOT / (
        "map_evaluation_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    map_results, map_details = evaluate_map_methods(
        map_manifest, map_output, settings=MapSettings(alpha=0.05), show_first=8,
    )
    display(map_results)
    print("CSVs, curvas PR, figuras e máscaras:", map_output)
